In [ ]:
import json
import os
import re
from pathlib import Path
from collections import defaultdict

# ── 1. 自动生成 mapping JSON ──
msa_temp = 'msa_template'

pdb_list_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/dimer/PDB.list"
with open(pdb_list_path) as f:
    ref_ids = {Path(line.strip()).stem for line in f if line.strip()}

output_base = f"/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/predict/original_result/{msa_temp}/output"
mapping = {}
for pred in sorted(os.listdir(output_base)):
    pred_path = Path(output_base) / pred
    if not pred_path.is_dir():
        continue
    parent_complex = pred.split("_")[0] if "_" in pred else pred
    if parent_complex in ref_ids:
        mapping[pred] = parent_complex
    else:
        print(f"警告：设计目录 '{pred}' 的 parent_complex '{parent_complex}' 不在 ref PDB 列表中，已跳过")

mapping_json = f"{msa_temp}_mapping.json"
with open(mapping_json, "w") as f:
    json.dump(mapping, f, indent=2)
print(f"生成 mapping JSON: {len(mapping)} 个设计目录映射 → {mapping_json}")

# ── 2. 运行指标计算 ──
# PYTHONUNBUFFERED=1 确保子进程的进度信息实时显示
os.system("PYTHONUNBUFFERED=1 /QIN/junjiechen/miniforge3/envs/Dpepalign/bin/python pipeline_metrics_v2.py \
            --mapping-json " + mapping_json + " \
            --ref-pdb-base /home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/dimer \
            --output-base /home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/predict/original_result/" + msa_temp + "/output \
            --peptide-chain-ref L \
            --peptide-chain-pred B \
            --num-workers 5 \
            --progress-every 10")

In [ ]:
# 将/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs中已经完成指标计算的目录移动到/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs-done
import os
import shutil
output_base = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs"
output_done_base = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs-done"
csv_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/metrics_summary.csv"
with open(csv_path, "r") as f:
    lines = f.readlines()
done_dirs = set()
for line in lines[1:]:
    dir_name = line.strip().split(",")[0]
    done_dirs.add(dir_name)

all_done = True
for dir_name in sorted(os.listdir(output_done_base)):
    if dir_name in done_dirs:
        continue
    else:
        print(f"{dir_name} not done yet.")
        all_done = False
        break
if all_done:
    print("All done! You can move the directories to the done folder.")



All done! You can move the directories to the done folder.


In [8]:
# 检查csv中complex的个数，是否每个complex都有5个seed，每个seed都有5个id
import pandas as pd
csv_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/metrics_summary.csv"
df = pd.read_csv(csv_path)
complex_counts = df['complex'].value_counts()
print(complex_counts)
for complex_name, count in complex_counts.items():
    if count != 25:
        print(f"{complex_name} has {count} entries, not 25.")

df['target'] = df['complex'].apply(lambda x: x.split("_")[0])
target = set(df['target'])
print(f"Total targets: {len(target)}")
    

complex
1a0n_1      25
1a0n_10     25
1a0n_100    25
1a0n_11     25
1a0n_12     25
            ..
6h7b_5      25
6h7b_6      25
6h7b_7      25
6h7b_8      25
6h7b_9      25
Name: count, Length: 8086, dtype: int64
Total targets: 110


In [2]:
import pandas as pd
# 读取/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs/metrics_summary.csv
# 针对每个complex

# 新抽样方式：样本中scRMSD、ipTM和pLDDT均小于THRESHOLDS的样本作为假阳性样本，统计每个靶点符合的样本数量
THRESHOLDS = {
    "scRMSD": 2.5,
    "pep_plddt": 70.0,
    "iptm": 0.7,
}

af_df = pd.read_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/outputs/metrics_summary.csv")
def count_false_positives(df: pd.DataFrame) -> pd.DataFrame:
    conditions = (
        (df["scRMSD"] <= THRESHOLDS["scRMSD"]) &
        (df["pep_plddt"] < THRESHOLDS["pep_plddt"]) &
        (df["iptm"] < THRESHOLDS["iptm"])
    )
    df["is_false_positive"] = conditions.astype(int)
    false_positive_counts = df.groupby("complex")["is_false_positive"].sum().reset_index(name="false_positive_count")
    return false_positive_counts

def count_true_positives(df: pd.DataFrame) -> pd.DataFrame:
    conditions = (
        (df["scRMSD"] <= THRESHOLDS["scRMSD"]) &
        (df["pep_plddt"] >= THRESHOLDS["pep_plddt"]) &
        (df["iptm"] >= THRESHOLDS["iptm"])
    )
    df["is_true_positive"] = conditions.astype(int)
    true_positive_counts = df.groupby("complex")["is_true_positive"].sum().reset_index(name="true_positive_count")
    return true_positive_counts

def count_false_positives_loosen(df: pd.DataFrame) -> pd.DataFrame:
    conditions = (
        (df["scRMSD"] <= THRESHOLDS["scRMSD"]) &
        (
            (df["pep_plddt"] < THRESHOLDS["pep_plddt"] ) |
            (df["iptm"] < THRESHOLDS["iptm"])
        )
    )
    df["is_false_positive_loosen"] = conditions.astype(int)
    false_positive_counts_loosen = df.groupby("complex")["is_false_positive_loosen"].sum().reset_index(name="false_positive_count_loosen")
    return false_positive_counts_loosen

false_positive_counts_df = count_false_positives(af_df)
false_positive_counts_df.to_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/af3_scrmsd_lower_pepplddt_lower_iptm_lower_counts_per_target.csv", index=False)
true_positive_counts_df = count_true_positives(af_df)
true_positive_counts_df.to_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/af3_scrmsd_lower_pepplddt_higher_iptm_higher_counts_per_target.csv", index=False)
false_positive_counts_loosen_df = count_false_positives_loosen(af_df)
false_positive_counts_loosen_df.to_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/af3_scrmsd_lower_pepplddt_lower_or_iptm_lower_counts_per_target.csv", index=False)

In [3]:
# 每个complex按最小add_distance_to_threshold选3个样本（scRMSD<2.5, pep_plddt<70或iptm<0.7）
# 每个target（complex名前4字符）最多取100个样本

import pandas as pd
import numpy as np

THRESHOLDS = {
    "scRMSD": 2.5,
    "pep_plDDT": 70.0,
    "ipTM": 0.7,
}


def add_distance_to_threshold(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["d_threshold"] = (
        ((out["scRMSD"] - THRESHOLDS["scRMSD"]).abs() / THRESHOLDS["scRMSD"])
        + ((out["pep_plDDT"] - THRESHOLDS["pep_plDDT"]).abs() / THRESHOLDS["pep_plDDT"])
        + ((out["ipTM"] - THRESHOLDS["ipTM"]).abs() / THRESHOLDS["ipTM"])
    )
    return out


# 读取CSV
csv_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/metrics_summary.csv"
df = pd.read_csv(csv_path)

# 计算d_threshold
df = add_distance_to_threshold(df)

# 每个complex: scRMSD<2.5 且 (pep_plddt<70 or iptm<0.7), 选d_threshold最小的3个
selected_list = []
for complex_name, grp in df.groupby("complex"):
    candidates = grp[
        (grp["scRMSD"] < THRESHOLDS["scRMSD"])
        & ((grp["pep_plDDT"] < THRESHOLDS["pep_plDDT"]) | (grp["ipTM"] < THRESHOLDS["ipTM"]))
    ]
    if len(candidates) > 0:
        selected_list.append(candidates.nsmallest(3, "d_threshold"))

selected = pd.concat(selected_list, ignore_index=True)

# 提取target（前4字符），每个target最多100个
selected["target"] = selected["complex"].str[:4]
final_list = []
for target_name, grp in selected.groupby("target"):
    final_list.append(grp.nsmallest(100, "d_threshold"))

final = pd.concat(final_list, ignore_index=True)

# 统计
print(f"原始样本数: {len(df)}")
n_qualified = (
    (df["scRMSD"] < THRESHOLDS["scRMSD"])
    & ((df["pep_plDDT"] < THRESHOLDS["pep_plDDT"]) | (df["ipTM"] < THRESHOLDS["ipTM"]))
).sum()
print(f"符合条件的样本 (scRMSD<2.5 且 pep_plDDT<70或ipTM<0.7): {n_qualified}")
print(f"每个complex选3个后: {len(selected)}")
print(f"每个target上限100后: {len(final)}")
print(f"涉及target数: {final['target'].nunique()}")
print(f"涉及complex数: {final['complex'].nunique()}")

per_complex = final.groupby("complex").size()
print(f"\n每个complex样本数: 最大={per_complex.max()}, 最小={per_complex.min()}")
per_target = final.groupby("target").size()
print(f"每个target样本数: 最大={per_target.max()}, 最小={per_target.min()}")

# 验证筛选条件
assert (final["scRMSD"] < THRESHOLDS["scRMSD"]).all(), "存在scRMSD>=2.5的样本!"
assert ((final["pep_plDDT"] < THRESHOLDS["pep_plDDT"]) | (final["ipTM"] < THRESHOLDS["ipTM"])).all(), "存在pep_plDDT>=70且ipTM>=0.7的样本!"
assert per_target.max() <= 100, f"存在target超过100个样本! 最大={per_target.max()}"
assert per_complex.max() <= 3, f"存在complex超过3个样本! 最大={per_complex.max()}"
print("\n质量检查全部通过!")

# 导出
output_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/selected_samples.csv"
final.to_csv(output_path, index=False)
print(f"\n已导出: {output_path}")

原始样本数: 202150
符合条件的样本 (scRMSD<2.5 且 pep_plDDT<70或ipTM<0.7): 15991
每个complex选3个后: 4563
每个target上限100后: 3527
涉及target数: 83
涉及complex数: 1534

每个complex样本数: 最大=3, 最小=1
每个target样本数: 最大=100, 最小=1

质量检查全部通过!

已导出: /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/AF3/predict/metrices/selected_samples.csv
